<a href="https://colab.research.google.com/github/kw3s/Colab/blob/main/YuE2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title 1. Check the GPU, VRAM, RAM and disk
!nvidia-smi
import os, shutil, sys
import torch

print("Python :", sys.version.split()[0])
print("Torch  :", torch.__version__)
print("CUDA   :", torch.cuda.is_available())
print("Disk   : %.1f GB free" % (shutil.disk_usage("/content").free / 1024 ** 3))

if not torch.cuda.is_available():
    raise SystemExit("No GPU. Runtime > Change runtime type > T4 GPU.")

props = torch.cuda.get_device_properties(0)
VRAM = props.total_memory / 1024 ** 3

try:
    RAM = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1024 ** 3
except (ValueError, OSError, AttributeError):
    RAM = 12.7

print("GPU    : %s (%.1f GB VRAM)" % (props.name, VRAM))
print("RAM    : %.1f GB" % RAM)

# Budget sizing. The 6.8 GiB model lives in VRAM, so VRAM is the binding term, not
# system RAM (an earlier version of this cell used min(VRAM, RAM) - 4.0, which
# under-sized the budget to 8.67 GiB on a T4 - below the ~9.2 GiB actually needed:
# 6.8 model + ~1.0 KV cache + 0.5 VAE + overhead). Reserve ~2 GiB for the CUDA context.
BUDGET = max(6.0, VRAM - 2.0)

# VAE tile size is chosen from VRAM, not from the budget, so a large card is not
# forced onto small tiles. cli.py ties this to budget <= 12, which is right for a T4
# but would under-use a 24 GiB+ GPU.
VAE_FRAMES = 512 if VRAM < 20 else 1024
print()
print("suggested memory_budget_gib : %.1f" % BUDGET)
print("vae_core_frames             : %d" % VAE_FRAMES)
print()

# The project's own README states: "Linux, Python 3.10+, 24GB NVIDIA GPU with BF16
# support." A free Colab T4 has 16 GB and compute capability 7.5, which predates
# native BF16 (that needs sm_80/Ampere or newer). So the T4 is BELOW the documented
# hardware target in two ways. This may still run with the reduced budget and
# offload, or it may fail outright - nobody has verified it on a T4.
cap = torch.cuda.get_device_capability(0)
print("compute capability : sm_%d%d" % cap)
if cap[0] < 8:
    print("WARNING: sm_%d%d has no native BF16. The pipeline reports")
    print("         model_dtype='bfloat16', so expect slow emulation or failure.")
if VRAM < 24:
    print("WARNING: %.1f GB VRAM is below the documented 24 GB requirement." % VRAM)
    print("         Reduce the budget, enable OFFLOAD_AR in cell 5, and keep songs short.")

# --- bf16 preflight -----------------------------------------------------------
# config.json declares dtype "bfloat16", and a T4 is sm_75 - native BF16 needs
# sm_80+. Rather than discover this 20 minutes into a generation, test it now.
# This is the single most likely reason a T4 run fails.
print()
print("bf16 preflight:")
try:
    a = torch.randn(512, 512, dtype=torch.bfloat16, device="cuda")
    torch.matmul(a, a)
    torch.cuda.synchronize()
    print("  bf16 matmul: OK (may still be slow without tensor cores)")
    del a
except Exception as exc:
    print("  bf16 matmul FAILED:", type(exc).__name__, exc)
    print("  This GPU cannot run the model's declared bfloat16 dtype reliably.")
    print("  It is below the documented target (24 GB, BF16) - not a notebook bug.")


Sun Sep 20 10:15:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# @title 2. Install yue2-infer without breaking Colab's torch
import importlib.metadata as md
import subprocess, sys


def version(pkg):
    try:
        return md.version(pkg)
    except md.PackageNotFoundError:
        return None


print("before:", {p: version(p) for p in ("torch", "torchvision", "numpy")})

# pyproject.toml pins torch==2.10.0 exactly, so a dependency-resolving install would
# replace Colab's CUDA-matched torch. Install the other dependencies explicitly first,
# then the wheel itself with --no-deps.
DEPS = [
    "transformers==4.57.6",
    "huggingface-hub==0.36.2",
    "safetensors==0.7.0",
    "tiktoken==0.12.0",
    "soundfile==0.13.1",
    "accelerate==1.13.0",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *DEPS], check=True)

# yue2-infer is NOT on PyPI - `pip install yue2-infer` fails with "from versions:
# none". The project ships prebuilt wheels on the model repo, and its README installs
# one by filename. Discover the newest instead of hardcoding a version that goes stale.
# (Imported after the pinned huggingface-hub is in place, for a consistent API.)
from huggingface_hub import hf_hub_download, list_repo_files

REPO = "m-a-p/YuE2-3B"
wheels = sorted(f for f in list_repo_files(REPO) if f.endswith(".whl"))
if not wheels:
    raise SystemExit("No yue2_infer wheel found in %s" % REPO)
wheel_name = wheels[-1]
print("available wheels:", wheels)
print("installing       :", wheel_name)

wheel_path = hf_hub_download(repo_id=REPO, filename=wheel_name)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
                wheel_path], check=True)

print("after :", {p: version(p) for p in ("torch", "torchvision", "numpy")})
print("yue2  :", version("yue2-infer"))

# --- confirm torch survived and the package imports --------------------------
import torch
print("torch.cuda.is_available():", torch.cuda.is_available())

try:
    import yue2
    print("yue2 imported OK, version", yue2.__version__)
except Exception as exc:
    print("IMPORT FAILED:", type(exc).__name__, exc)
    print()
    print("If it failed, try installing from the source tree instead:")
    print("    !git clone --depth 1 "
          "https://github.com/multimodal-art-projection/YuE /content/YuE")
    print("    !pip install -q --no-deps /content/YuE")
    print("then Runtime > Restart session and re-run from cell 1.")


before: {'torch': '2.11.0+cu128', 'torchvision': '0.26.0+cu128', 'numpy': '2.1.3'}


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


available wheels: ['yue2_infer-0.1.3-py3-none-any.whl', 'yue2_infer-0.1.5-py3-none-any.whl']
installing       : yue2_infer-0.1.5-py3-none-any.whl
after : {'torch': '2.11.0+cu128', 'torchvision': '0.26.0+cu128', 'numpy': '2.1.3'}
yue2  : 0.1.5
torch.cuda.is_available(): True
yue2 imported OK, version 0.1.5


In [ ]:
# @title 3. Download the YuE2 weights
import os

# Optional: a free HF read token lifts anonymous rate limits considerably on a
# 7 GB download. Add it via the key icon in the left sidebar, then uncomment.
# from google.colab import userdata
# os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

from huggingface_hub import snapshot_download

MODEL_REPO = "m-a-p/YuE2-3B"      # model.safetensors is ~6.8 GB
VAE_REPO = "m-a-p/YuE2-Vae"       # separate decoder repo; the pipeline needs both

for repo, why in ((MODEL_REPO, "3B model"), (VAE_REPO, "decoder VAE")):
    print("fetching %s (%s) ..." % (repo, why))
    try:
        path = snapshot_download(repo_id=repo)
        print("  ->", path)
    except Exception as exc:
        print("  FAILED: %s: %s" % (type(exc).__name__, exc))
        if repo == VAE_REPO:
            print()
            print("  The VAE has more than one release. If 'standard' fails, the")
            print("  pipeline also accepts 'legacy' or an explicit repo id, e.g.")
            print("      vae='m-a-p/YuE2-Vae-legacy'")
            print("  or point it at a local directory containing the VAE files.")
        else:
            raise

print()
print("Cached HF downloads:")
root = os.path.expanduser("~/.cache/huggingface/hub")
if os.path.isdir(root):
    for name in sorted(os.listdir(root)):
        if name.startswith("models--m-a-p"):
            print("  ", name)


fetching m-a-p/YuE2-3B (3B model) ...


Fetching 28 files:   0%|          | 0/28 [00:00<?, ?it/s]

  -> /root/.cache/huggingface/hub/models--m-a-p--YuE2-3B/snapshots/14fc6c6f146441b1dd6363fcb2e01e82a6914cb7
fetching m-a-p/YuE2-Vae (decoder VAE) ...


Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

  -> /root/.cache/huggingface/hub/models--m-a-p--YuE2-Vae/snapshots/9a94e1d0ea9f8087e98f77fa88df4a4068104d2a

Cached HF downloads:
   models--m-a-p--YuE2-3B
   models--m-a-p--YuE2-Vae


In [ ]:
# @title 4. Environment report (the package's own `yue2 doctor`)
import subprocess, sys

cmd = [sys.executable, "-m", "yue2.cli", "doctor",
       "--budget", str(BUDGET), "--model", "m-a-p/YuE2-3B",
       "--vae", "m-a-p/YuE2-Vae"]
print(" ".join(cmd))
print("-" * 72)
proc = subprocess.run(cmd, capture_output=True, text=True)
print(proc.stdout or "(no stdout)")
if proc.returncode != 0:
    print("stderr:")
    print(proc.stderr or "(none)")
    print()
    print("A non-zero exit here usually means a missing dependency or no GPU.")
else:
    print("doctor exited 0.")


/usr/bin/python3 -m yue2.cli doctor --budget 12.56317138671875 --model m-a-p/YuE2-3B --vae m-a-p/YuE2-Vae
------------------------------------------------------------------------
{
  "dependencies_ready": true,
  "versions": {
    "torch": "2.11.0+cu128",
    "transformers": "4.57.6",
    "huggingface-hub": "0.36.2",
    "safetensors": "0.7.0",
    "tiktoken": "0.12.0",
    "soundfile": "0.13.1"
  },
  "cuda": [
    {
      "id": 0,
      "name": "Tesla T4",
      "memory_gib": 14.56317138671875,
      "compute_capability": [
        7,
        5
      ]
    }
  ],
  "mps_available": false,
  "model": "m-a-p/YuE2-3B",
  "vae": "m-a-p/YuE2-Vae",
  "default_cot": "full",
  "default_cfg": {
    "full": 1.0,
    "melody": 1.0,
    "off": 1.01
  },
  "validated": false,
  "note": "Environment readiness is not quality or real-24GB acceptance."
}

doctor exited 0.


In [ ]:
import subprocess, sys
print(subprocess.run([sys.executable, "-m", "pip", "show", "yue2-infer"],
                     capture_output=True, text=True).stdout or "NOT INSTALLED")


NOT INSTALLED


In [ ]:
# @title 4c. Pre-Ampere attention patch (RUN BEFORE CELL 5) - needed on a T4
# Error without this: "RuntimeError: FlashAttention only supports Ampere GPUs or
# newer." A T4 is sm_75 (Turing); FlashAttention needs sm_80+.
#
# Cause, from the released modeling_yue2.py (sdpa(), lines ~20-32):
#     return F.scaled_dot_product_attention(
#         query, key, value, attn_mask=attn_mask, is_causal=is_causal,
#         enable_gqa=grouped,          # <-- only the FlashAttention backend supports GQA
#     )
# enable_gqa requires the flash backend, which does not exist below sm_80, so there
# is no backend left and PyTorch aborts.
#
# Locally the model already does exactly the right thing for MPS: expand the KV
# heads so no GQA is needed, then call SDPA without enable_gqa. The same fallback is
# valid for pre-Ampere CUDA, and that is what this cell installs.
#
# NOT upstream-supported. The project documents a 24 GB BF16 GPU, so this is at your
# own risk. It changes the attention kernel, not the math.
import torch

if not torch.cuda.is_available():
    print("No CUDA; nothing to patch.")
else:
    cap = torch.cuda.get_device_capability(0)

    if cap[0] >= 8:
        print("sm_%d%d: FlashAttention is supported here, no patch needed." % cap)
    else:
        import yue2.modeling_yue2 as yue2_modeling

        def sdpa_pre_ampere(query, key, value, *, attn_mask=None, is_causal=False):
            # Same strategy as the upstream MPS path: materialise the grouped KV
            # heads, so enable_gqa is unnecessary and a plain math backend works.
            if query.shape[1] != key.shape[1]:
                groups = query.shape[1] // key.shape[1]
                key = key.repeat_interleave(groups, dim=1)
                value = value.repeat_interleave(groups, dim=1)
            return torch.nn.functional.scaled_dot_product_attention(
                query, key, value, attn_mask=attn_mask, is_causal=is_causal)

        yue2_modeling.sdpa = sdpa_pre_ampere
        print("sm_%d%d: patched yue2.modeling_yue2.sdpa for pre-Ampere CUDA." % cap)
        print()
        print("The patch takes effect when the model runs, so you do NOT need to")
        print("reload it - just continue to cell 5, then cell 6.")
        print()
        print("Sanity check (expect no exception):")
        try:
            q = torch.randn(1, 16, 8, 128, dtype=torch.bfloat16, device="cuda")
            k = torch.randn(1, 8, 8, 128, dtype=torch.bfloat16, device="cuda")
            v = torch.randn(1, 8, 8, 128, dtype=torch.bfloat16, device="cuda")
            out = sdpa_pre_ampere(q, k, v, is_causal=True)
            torch.cuda.synchronize()
            print("  ok, output shape", tuple(out.shape))
        except Exception as exc:
            print("  FAILED:", type(exc).__name__, exc)


sm_75: patched yue2.modeling_yue2.sdpa for pre-Ampere CUDA.

The patch takes effect when the model runs, so you do NOT need to
reload it - just continue to cell 5, then cell 6.

Sanity check (expect no exception):
  ok, output shape (1, 16, 8, 128)


In [ ]:
# @title 5. Load the YuE2 pipeline (with import diagnostics)
import gc, sys, sysconfig, time
import torch

# Optional switches, sized for a free T4 (16 GB VRAM / ~12.7 GB RAM):
#   offload_ar     - move the autoregressive weights to CPU between stages, trading
#                    speed for peak memory. Turn this ON if you hit CUDA OOM.
#   quantization   - "none" or "fp8". fp8 needs Ada/Hopper (sm_89+); a T4 is sm_75,
#                    so leave this as "none".
OFFLOAD_AR = False
QUANTIZATION = "none"

# --- import diagnostics -------------------------------------------------------
# A common failure: cell 4's `doctor` runs the CLI in a SUBPROCESS, which succeeds,
# while this kernel still cannot import yue2. Subprocess success does not prove the
# kernel sees the package, so check explicitly and explain what to do.
try:
    from yue2 import YuE2Pipeline   # documented import path
    print("imported yue2 from:", sys.modules["yue2"].__file__)
except ModuleNotFoundError as exc:
    print("ModuleNotFoundError:", exc)
    print()
    print("kernel python     :", sys.executable)
    print("kernel purelib    :", sysconfig.get_paths()["purelib"])

    # Is it visible to the interpreter that runs pip, but not to this kernel?
    import subprocess
    probe = subprocess.run(
        [sys.executable, "-c", "import yue2, sys; print(yue2.__file__)"],
        capture_output=True, text=True)
    print("subprocess probe  :", (probe.stdout or probe.stderr).strip() or "(no output)")

    print()
    print("Most likely cause: cell 2 ran `pip`, but the wheel went to a different")
    print("interpreter than this kernel. After ANY install that adds a new package,")
    print("Colab's kernel caches sys.modules - you must restart the runtime:")
    print()
    print("    Runtime > Restart session")
    print()
    print("Then re-run cells 1, 3, 5, 6, 7, 8, 9. Cell 2 is NOT needed again unless")
    print("a restart wiped it; if `import yue2` fails after the restart too, re-run")
    print("cell 2 and restart once more.")
    print()
    print("Verify quickly in a fresh cell, no restart required:")
    print("    import subprocess, sys")
    print("    print(subprocess.run([sys.executable, '-m', 'pip', 'show', 'yue2-infer'],")
    print("                         capture_output=True, text=True).stdout)")
    raise

if "pipe" in globals():
    try:
        pipe.close()
    except Exception:
        pass
    del pipe
    gc.collect()
    torch.cuda.empty_cache()

t0 = time.time()
pipe = YuE2Pipeline.from_pretrained(
    "m-a-p/YuE2-3B",
    vae="m-a-p/YuE2-Vae",
    device="auto",
    memory_budget_gib=BUDGET,
    backend="torch",
    quantization=QUANTIZATION,
    offload_ar=OFFLOAD_AR,
    # Tile size derived from VRAM in cell 1 (512 on a T4, 1024 on 20 GiB+).
    vae_core_frames=VAE_FRAMES,
    progress=True,
)
print("loaded in %.1f s" % (time.time() - t0))
print("device            :", pipe.device)
print("memory_budget_gib :", pipe.memory_budget_gib)
print("vae_core_frames   :", pipe.vae_core_frames)

if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print("VRAM free/total   : %.1f / %.1f GB" % (free / 1024**3, total / 1024**3))


[YuE2] Starting Resolving model files: elapsed 0.0s


imported yue2 from: /usr/local/lib/python3.13/dist-packages/yue2/__init__.py


Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

[YuE2] Completed Resolving model files: elapsed 0.1s
[YuE2] Starting Verifying model files: elapsed 0.0s
[YuE2] Running Verifying model files: elapsed 5.0s
[YuE2] Running Verifying model files: elapsed 10.0s
[YuE2] Running Verifying model files: elapsed 15.0s
[YuE2] Running Verifying model files: elapsed 20.0s
[YuE2] Running Verifying model files: elapsed 25.0s
[YuE2] Running Verifying model files: elapsed 30.0s
[YuE2] Running Verifying model files: elapsed 35.0s
[YuE2] Running Verifying model files: elapsed 40.0s
[YuE2] Completed Verifying model files: elapsed 41.0s


loaded in 41.4 s
device            : cuda:0
memory_budget_gib : 12.56317138671875
vae_core_frames   : 512
VRAM free/total   : 14.4 / 14.6 GB


In [ ]:
return F.scaled_dot_product_attention(
    query, key, value, attn_mask=attn_mask, is_causal=is_causal,
    enable_gqa=grouped,          # only the FlashAttention backend supports GQA
)


SyntaxError: 'return' outside function (1129031869.py, line 1)

In [ ]:
# @title 4c. Pre-Ampere attention patch (run before cell 5) - required on a T4
#
# Without this, generation raises: "FlashAttention only supports Ampere GPUs or
# newer." A T4 is sm_75 (Turing); FlashAttention needs sm_80 or newer.
#
# Why it happens: the released model calls scaled_dot_product_attention with
# enable_gqa=True. Only the FlashAttention backend implements grouped-query
# attention, so below sm_80 every candidate backend is rejected and PyTorch raises
# instead of quietly degrading to a slower kernel.
#
# The fix: expand the grouped KV heads so GQA is unnecessary, then call SDPA
# plainly. This mirrors the fallback the model already ships for MPS, just applied
# to older CUDA devices. It changes the attention kernel, not the arithmetic.
#
# Not upstream-supported. The project documents a 24GB BF16 GPU. Run it only if
# cell 1 reported sm_75.
import torch

if not torch.cuda.is_available():
    print("No CUDA; nothing to patch.")
else:
    cap = torch.cuda.get_device_capability(0)

    if cap[0] >= 8:
        print("sm_%d%d: FlashAttention is supported here, no patch needed." % cap)
    else:
        import yue2.modeling_yue2 as yue2_modeling

        def sdpa_pre_ampere(query, key, value, *, attn_mask=None, is_causal=False):
            # Materialise the grouped KV heads, as the model's own MPS path does,
            # so enable_gqa is not needed and a plain kernel will run.
            if query.shape[1] != key.shape[1]:
                groups = query.shape[1] // key.shape[1]
                key = key.repeat_interleave(groups, dim=1)
                value = value.repeat_interleave(groups, dim=1)
            return torch.nn.functional.scaled_dot_product_attention(
                query, key, value, attn_mask=attn_mask, is_causal=is_causal)

        yue2_modeling.sdpa = sdpa_pre_ampere
        print("sm_%d%d: patched yue2.modeling_yue2.sdpa for pre-Ampere CUDA." % cap)
        print()
        print("Model modules call sdpa() by name at run time, so this rebind applies")
        print("without reloading. Continue to cell 5, then cell 6.")
        print()
        print("Sanity check with the model's real shapes (16 q heads, 8 kv heads):")
        try:
            q = torch.randn(1, 16, 8, 128, dtype=torch.bfloat16, device="cuda")
            k = torch.randn(1, 8, 8, 128, dtype=torch.bfloat16, device="cuda")
            v = torch.randn(1, 8, 8, 128, dtype=torch.bfloat16, device="cuda")
            out = sdpa_pre_ampere(q, k, v, is_causal=True)
            torch.cuda.synchronize()
            print("  ok, output shape", tuple(out.shape))
        except Exception as exc:
            print("  FAILED:", type(exc).__name__, exc)


In [ ]:
# @title 7. Listen - play the result inline and list the artifacts
import glob, os
import numpy as np
import soundfile as sf
from IPython.display import Audio, FileLink, display

OUT = SAVE_TO
files = sorted(glob.glob(os.path.join(OUT, "*")))
print("artifacts in %s:" % OUT)
for f in files:
    print("   %-24s %9.2f KB" % (os.path.basename(f), os.path.getsize(f) / 1024))

# result.audio is float32 clamped to [-1, 1]. soundfile wants (samples,) or
# (samples, channels), so normalise whichever layout the VAE produced.
a = np.asarray(result.audio)
if a.ndim == 2 and a.shape[0] <= 2 and a.shape[1] > 2:
    a = a.T                      # (channels, samples) -> (samples, channels)
elif a.ndim == 1:
    a = a[:, None]
a = a.astype("float32")

wav = os.path.join(OUT, "song.wav")
sf.write(wav, a, result.sample_rate)
print()
print("wrote %s (%.1f s, %d Hz)" % (wav, len(a) / result.sample_rate, result.sample_rate))

display(Audio(filename=wav))
display(FileLink(wav))


In [ ]:
# @title 8. Inspect the symbolic plan (YuE2's editable composition step)
from yue2.pipeline import SymbolicPlan

print("from the in-memory result:")
print("  cot mode   :", result.semantic.plan.request.cot)
print("  abc tokens :", len(result.semantic.plan.abc_ids))
print("  truncated  :", result.truncated)
print()

# Reload from disk to prove the saved artifacts are self-contained. SymbolicPlan.load
# verifies the sha256 manifest, so this also confirms nothing was corrupted.
try:
    plan = SymbolicPlan.load(SAVE_TO)
    print("reloaded from %s:" % SAVE_TO)
    print("  truncated  :", plan.truncated)
    print("  abc tokens :", len(plan.abc_ids))
    print()
    if plan.abc:
        text = plan.abc
        print(text[:1500] + ("\n... [%d more chars]" % (len(text) - 1500)
                             if len(text) > 1500 else ""))
    else:
        print("(no ABC text - cot mode 'off' skips planning entirely)")
except Exception as exc:
    print("could not reload plan:", type(exc).__name__, exc)
